# `trade_levels.py` — Playground

Manual verification notebook for **shared trade-level and EV helpers** used by Gate 5.

| Function | Status | Notes |
|---|---|---|
| `build_trade_levels(candidate)` | ✅ built | Stop/target from price + ATR (fixed ~2:1 R:R) |
| `estimate_win_probability(candidate, gate_results)` | ✅ built | Rules-only mapping from momentum + Gate 3 |
| `calculate_expected_value(win_prob, reward_risk)` | ✅ built | `EV = (p × R) − (1 − p)` |
| `build_gate_summary(gate_results)` | ✅ built | Audit text for Gate 1–4 outputs |

**No LLM, no API keys.** Tune `WIN_PROB_*` constants in `trade_levels.py` from paper-trading data during weekly review.

In [ ]:
import sys
import pathlib

logic_dir = pathlib.Path('.').resolve()
if not (logic_dir / 'trade_levels.py').exists():
    logic_dir = pathlib.Path('backend/02_intelligence/helpers/logic').resolve()

intelligence_dir = logic_dir.parent.parent
if str(intelligence_dir) not in sys.path:
    sys.path.insert(0, str(intelligence_dir))

from helpers.logic.trade_levels import (
    build_trade_levels,
    estimate_win_probability,
    calculate_expected_value,
    build_gate_summary,
)

---
## Happy path — trade levels for a momentum candidate

Expect ~2:1 reward:risk, stop at entry − 1.5×ATR, target at 2× stop distance.

In [ ]:
candidate = {'price': 875.50, 'atr': 12.30, 'score': 3}
build_trade_levels(candidate)

---
## Variation — strong vs weak win-probability mapping

Strong: score 3 + BULLISH conf 9. Weak: score 2 + BULLISH conf 6 with caution. Expect strong EV >> weak EV.

In [ ]:
levels = build_trade_levels(candidate)
rr = levels['reward_risk']

strong_gates = {'gate3': {'direction': 'BULLISH', 'confidence': 9, 'caution': False}}
weak_gates   = {'gate3': {'direction': 'BULLISH', 'confidence': 6, 'caution': True}}

for label, score, gates in [('strong', 3, strong_gates), ('weak', 2, weak_gates)]:
    p = estimate_win_probability({'score': score}, gates)
    ev = calculate_expected_value(p, rr)
    print(f'{label:6} score={score} win_prob={p:.0%} EV={ev:.3f}')

---
## Gate summary — audit block for logging

In [ ]:
print(build_gate_summary({
    'gate1': {'passed': True},
    'gate2': {'passed': True},
    'gate3': {'direction': 'BULLISH', 'confidence': 9, 'caution': False, 'key_reason': 'Strong demand'},
    'gate4': {'action': 'PASS', 'contradiction_type': 'none', 'risk_level': 'NONE', 'reason': 'NONE'},
}))

---
## Failure path — missing price/atr

Caller must supply `price` and `atr`; missing keys raise `KeyError`.

In [ ]:
try:
    build_trade_levels({'ticker': 'BAD'})
except KeyError as e:
    print(f'Expected KeyError: {e}')

---
## Free-play